Here we train the model that that will be used in the deployment.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import make_pipeline
from sklearn import linear_model
from sklearn.metrics import roc_auc_score

import pickle

In [20]:
df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df = df.rename(columns=lambda x: x.lower().replace(' ', '_'))
#Newer versions of pandas moved from 'object' to 'string', so this command prevents the mistake caused by version deprecation
text_columns = df.select_dtypes(include=['object', 'string']).columns

df[text_columns] = df[text_columns].apply(
    lambda col: col.str.lower().str.replace(' ', '_', regex=False)
)

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [23]:
df = df.drop(['customerid'], axis=1)

In [24]:
df.head()

,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,female,0,yes,no,1,no,no_phone_service,dsl,no,yes,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,0
1,male,0,no,no,34,yes,no,dsl,yes,no,yes,no,no,no,one_year,no,mailed_check,56.95,1889.50,0
2,male,0,no,no,2,yes,no,dsl,yes,yes,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,1
3,male,0,no,no,45,no,no_phone_service,dsl,yes,no,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,0
4,female,0,no,no,2,yes,no,fiber_optic,no,no,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,1


In [17]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

In [25]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

train_dict = df_full_train.drop(['churn'], axis=1).to_dict(orient='records')

test_dict = df_test.drop(['churn'], axis=1).to_dict(orient='records')

# Create a pipeline that includes a DictVectorizer and a Logistic Regression model.
# The DictVectorizer will convert the dictionary representation of the data into a format suitable for the model,
# and the Logistic Regression model will be used to predict churn.
pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    linear_model.LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
)

In [26]:
pipeline.fit(train_dict, df_full_train.churn)

# Compute the AUC score on the test set
y_test_pred = pipeline.predict_proba(test_dict)[:, 1]

auc = roc_auc_score(df_test.churn, y_test_pred)
print(f'Test AUC score: {auc:.3f}')

Test AUC score: 0.858


Now, we save the model into a binary file.

In [27]:
with open('LogReg_model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

Open the model file and check that it properly reproduces the model.

In [28]:
with open('LogReg_model.bin', 'rb') as f_in:
    pipe = pickle.load(f_in)

In [29]:
customer = df.iloc[3]
print(pipe.predict_proba([customer])[0,1], pipeline.predict_proba([customer])[0,1])

0.02478192157891284 0.02478192157891284


Run the following command to start the web service.

In [ ]:
uv run uvicorn customer_api:app --host 127.0.0.1 --port 9696

In [36]:
customer.drop(['churn']).to_dict()

{'gender': 'male',
 'seniorcitizen': 0,
 'partner': 'no',
 'dependents': 'no',
 'tenure': 45,
 'phoneservice': 'no',
 'multiplelines': 'no_phone_service',
 'internetservice': 'dsl',
 'onlinesecurity': 'yes',
 'onlinebackup': 'no',
 'deviceprotection': 'yes',
 'techsupport': 'yes',
 'streamingtv': 'no',
 'streamingmovies': 'no',
 'contract': 'one_year',
 'paperlessbilling': 'no',
 'paymentmethod': 'bank_transfer_(automatic)',
 'monthlycharges': 42.3,
 'totalcharges': 1840.75}

In [37]:
customer.churn

np.int64(0)